This code allows to combine recommeded manual tracks files into a unified dataset.

In [ ]:
import json
from pathlib import Path
import geopandas as gpd
from shapely.geometry import LineString

all_records = []
folder_path = Path("../data/manualRecommendedTracks")

for file_path in folder_path.glob('*.json'):
    with open(file_path, 'r', encoding='utf-8') as file:
        try:
            data = json.load(file)
            meta = data['tracks'][0]
    
            coords = meta.get("track")
            if coords is None:
                print(f"Skipping {file_path.name}: No 'tracks' key found.")
                continue
                
            track_geometry = LineString(coords)
            
            record = {
                "name": meta.get("name"),
                "description": meta.get("description"),
                "trackMapTargets": meta.get("trackMapTargets"),
                "depth": meta.get("depth"),
                "mapTarget": meta.get("mapTarget"),
                "center": meta.get("center"),
                "boundingBox": meta.get("boundingBox"),
                "geometry": track_geometry
            }
            
            all_records.append(record)
            
        except Exception as e:
            print(f"Error processing {file_path.name}: {e}")

if len(all_records) == 0:
    print("No data parsed.")
else:
    # Create GeoDataFrame
    gdf = gpd.GeoDataFrame(all_records, geometry='geometry', crs="EPSG:4326")
    gdf.to_file("recommended_tracks.gpkg", driver="GPKG")

In [ ]:
gdf = gdf.to_crs("EPSG:3006")

gdf.to_file("data/processed/recommended_tracks.gpkg", driver="GPKG")